In [1]:
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

df = pd.read_parquet(PROCESSED_DIR / "clean_transactions.parquet")

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df = df.rename(columns={
    "invoiceno": "invoice_no",
    "stockcode": "stock_code",
    "invoicedate": "invoice_date",
    "unitprice": "unit_price",
    "customerid": "customer_id"
})

df["invoice_date"] = pd.to_datetime(df["invoice_date"])

df_valid = df[df["is_valid_order_line"]].copy()

print(df_valid.columns.tolist())
print(df_valid.shape)

['invoice_no', 'stock_code', 'description', 'quantity', 'invoice_date', 'unit_price', 'customer_id', 'country', 'line_revenue_gross', 'is_cancelled_invoice', 'is_refund_or_return', 'is_positive_purchase', 'is_identified_customer', 'exclusion_reason', 'is_valid_order_line', 'analytical_revenue']
(397884, 16)


In [2]:
customer_orders = (
    df_valid.groupby(["customer_id", "invoice_no"])
    .agg(
        order_date=("invoice_date", "min"),
        order_revenue=("analytical_revenue", "sum")
    )
    .reset_index()
)

customer_orders = customer_orders[
    customer_orders["order_revenue"] > 0
].copy()

customer_orders = customer_orders.sort_values(["customer_id", "order_date", "invoice_no"])

customer_orders["order_rank"] = (
    customer_orders.groupby("customer_id").cumcount() + 1
)

customer_orders["is_first_order"] = customer_orders["order_rank"] == 1
customer_orders["is_repeat_order"] = customer_orders["order_rank"] > 1

customer_orders.head()

,customer_id,invoice_no,order_date,order_revenue,order_rank,is_first_order,is_repeat_order
0,12346,541431,2011-01-18 10:01:00,77183.60,1,True,False
1,12347,537626,2010-12-07 14:57:00,711.79,1,True,False
2,12347,542237,2011-01-26 14:30:00,475.39,2,False,True
3,12347,549222,2011-04-07 10:43:00,636.25,3,False,True
4,12347,556201,2011-06-09 13:01:00,382.52,4,False,True


In [3]:
customer_metrics = (
    customer_orders.groupby("customer_id")
    .agg(
        total_orders=("invoice_no", "nunique"),
        revenue=("order_revenue", "sum"),
        first_purchase=("order_date", "min"),
        last_purchase=("order_date", "max")
    )
    .reset_index()
)

customer_metrics["aov"] = customer_metrics["revenue"] / customer_metrics["total_orders"]

customer_metrics["active_days"] = (
    customer_metrics["last_purchase"] - customer_metrics["first_purchase"]
).dt.days

customer_metrics["is_repeat_customer"] = customer_metrics["total_orders"] > 1

customer_metrics.to_parquet(PROCESSED_DIR / "customer_metrics.parquet", index=False)

customer_metrics.head()

,customer_id,total_orders,revenue,first_purchase,last_purchase,aov,active_days,is_repeat_customer
0,12346,1,77183.60,2011-01-18 10:01:00,2011-01-18 10:01:00,77183.600000,0,False
1,12347,7,4310.00,2010-12-07 14:57:00,2011-12-07 15:52:00,615.714286,365,True
2,12348,4,1797.24,2010-12-16 19:09:00,2011-09-25 13:13:00,449.310000,282,True
3,12349,1,1757.55,2011-11-21 09:51:00,2011-11-21 09:51:00,1757.550000,0,False
4,12350,1,334.40,2011-02-02 16:01:00,2011-02-02 16:01:00,334.400000,0,False


In [4]:
first_order_revenue = customer_orders.loc[
    customer_orders["is_first_order"], "order_revenue"
].sum()

repeat_order_revenue = customer_orders.loc[
    customer_orders["is_repeat_order"], "order_revenue"
].sum()

one_time_customers = customer_metrics.loc[
    ~customer_metrics["is_repeat_customer"], "customer_id"
]

repeat_customers = customer_metrics.loc[
    customer_metrics["is_repeat_customer"], "customer_id"
]

one_time_revenue = customer_orders.loc[
    customer_orders["customer_id"].isin(one_time_customers),
    "order_revenue"
].sum()

repeat_customer_revenue = customer_orders.loc[
    customer_orders["customer_id"].isin(repeat_customers),
    "order_revenue"
].sum()

lifecycle = pd.DataFrame({
    "metric": [
        "first_order_revenue",
        "repeat_order_revenue",
        "one_time_customer_revenue",
        "repeat_customer_revenue"
    ],
    "value": [
        first_order_revenue,
        repeat_order_revenue,
        one_time_revenue,
        repeat_customer_revenue
    ]
})

lifecycle["share_of_identifiable_revenue"] = lifecycle["value"] / customer_orders["order_revenue"].sum()

lifecycle.to_csv(PROCESSED_DIR / "customer_lifecycle_revenue_split.csv", index=False)

lifecycle

,metric,value,share_of_identifiable_revenue
0,first_order_revenue,1846498.003,0.207206
1,repeat_order_revenue,7064909.901,0.792794
2,one_time_customer_revenue,616311.731,0.069160
3,repeat_customer_revenue,8295096.173,0.930840


In [5]:
second_orders = customer_orders[customer_orders["order_rank"] == 2].copy()

conversion_rate = len(second_orders) / len(customer_metrics)

days_to_second = second_orders.merge(
    customer_metrics[["customer_id", "first_purchase"]],
    on="customer_id",
    how="left"
)

days_to_second["days_to_second"] = (
    days_to_second["order_date"] - days_to_second["first_purchase"]
).dt.days

median_days = days_to_second["days_to_second"].median()

print(f"Second purchase conversion: {conversion_rate:.2%}")
print(f"Median days to second purchase: {median_days:.0f}")

Second purchase conversion: 65.58%
Median days to second purchase: 50


In [6]:
customer_metrics = customer_metrics.sort_values("revenue", ascending=False).copy()

customer_metrics["rank"] = range(1, len(customer_metrics) + 1)

customer_metrics["customer_decile"] = pd.qcut(
    customer_metrics["rank"],
    10,
    labels=[
        "Top 10%",
        "10-20%",
        "20-30%",
        "30-40%",
        "40-50%",
        "50-60%",
        "60-70%",
        "70-80%",
        "80-90%",
        "Bottom 10%"
    ]
)

deciles = (
    customer_metrics.groupby("customer_decile", observed=False)
    .agg(
        customers=("customer_id", "count"),
        revenue=("revenue", "sum")
    )
    .reset_index()
)

total_revenue = customer_metrics["revenue"].sum()

deciles["customer_pct"] = deciles["customers"] / len(customer_metrics)
deciles["revenue_pct"] = deciles["revenue"] / total_revenue

deciles.to_csv(PROCESSED_DIR / "customer_revenue_deciles.csv", index=False)

deciles

,customer_decile,customers,revenue,customer_pct,revenue_pct
0,Top 10%,434,5469382.460,0.100046,0.613751
1,10-20%,434,1180055.001,0.100046,0.132421
2,20-30%,434,728457.380,0.100046,0.081744
3,30-40%,433,490811.450,0.099816,0.055077
4,40-50%,434,344734.671,0.100046,0.038685
5,50-60%,434,252901.481,0.100046,0.028380
6,60-70%,433,179506.281,0.099816,0.020143
7,70-80%,434,132219.390,0.100046,0.014837
8,80-90%,434,87239.050,0.100046,0.009790
9,Bottom 10%,434,46100.740,0.100046,0.005173


In [7]:
customer_orders["order_month"] = customer_orders["order_date"].dt.to_period("M")

first_purchase_month = (
    customer_orders.groupby("customer_id")["order_month"]
    .min()
    .rename("cohort_month")
)

customer_orders = customer_orders.merge(
    first_purchase_month,
    on="customer_id",
    how="left"
)

customer_orders["month_index"] = (
    customer_orders["order_month"].astype(int)
    - customer_orders["cohort_month"].astype(int)
)

cohort_counts = (
    customer_orders.groupby(["cohort_month", "month_index"])
    .agg(customers=("customer_id", "nunique"))
    .reset_index()
)

cohort_sizes = (
    cohort_counts[cohort_counts["month_index"] == 0]
    [["cohort_month", "customers"]]
    .rename(columns={"customers": "cohort_size"})
)

retention = cohort_counts.merge(cohort_sizes, on="cohort_month", how="left")
retention["retention_pct"] = retention["customers"] / retention["cohort_size"]

retention_matrix = retention.pivot_table(
    index="cohort_month",
    columns="month_index",
    values="retention_pct"
)

retention_matrix.to_csv(PROCESSED_DIR / "retention_matrix.csv")

retention_matrix

month_index,0,1,2,3,4,5,6,7,8,9,10,11,12
cohort_month,,,,,,,,,,,,,
2010-12,1.0,0.366102,0.323164,0.384181,0.362712,0.397740,0.362712,0.349153,0.353672,0.395480,0.374011,0.502825,0.265537
2011-01,1.0,0.220624,0.266187,0.230216,0.321343,0.287770,0.247002,0.242206,0.299760,0.326139,0.364508,0.117506,NaN
2011-02,1.0,0.186842,0.186842,0.284211,0.271053,0.247368,0.252632,0.278947,0.247368,0.305263,0.068421,NaN,NaN
2011-03,1.0,0.150442,0.252212,0.199115,0.223451,0.168142,0.267699,0.230088,0.278761,0.086283,NaN,NaN,NaN
2011-04,1.0,0.213333,0.203333,0.210000,0.196667,0.226667,0.216667,0.260000,0.073333,NaN,NaN,NaN,NaN
2011-05,1.0,0.190141,0.172535,0.172535,0.207746,0.232394,0.264085,0.095070,NaN,NaN,NaN,NaN,NaN
2011-06,1.0,0.173554,0.157025,0.264463,0.231405,0.334711,0.095041,NaN,NaN,NaN,NaN,NaN,NaN
2011-07,1.0,0.180851,0.207447,0.223404,0.271277,0.111702,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2011-08,1.0,0.207101,0.248521,0.242604,0.124260,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
import subprocess
from pathlib import Path
from datetime import datetime

NOTEBOOK_PATH = Path("02_retention_analysis.ipynb")
LOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = LOG_DIR / f"notebook_output_{timestamp}.md"

cmd = [
    "jupyter", "nbconvert",
    "--to", "markdown",
    str(NOTEBOOK_PATH),
    "--stdout"
]

with open(output_file, "w", encoding="utf-8") as f:
    result = subprocess.run(cmd, stdout=f, stderr=subprocess.PIPE, text=True)

print(f"Notebook exported to: {output_file}")

Notebook exported to: logs/notebook_output_20260504_174032.md


In [9]:
print(lifecycle)
print(deciles)
print(f"Second purchase conversion: {conversion_rate:.2%}")
print(f"Median days to second purchase: {median_days:.0f}")

                      metric        value  share_of_identifiable_revenue
0        first_order_revenue  1846498.003                       0.207206
1       repeat_order_revenue  7064909.901                       0.792794
2  one_time_customer_revenue   616311.731                       0.069160
3    repeat_customer_revenue  8295096.173                       0.930840
  customer_decile  customers      revenue  customer_pct  revenue_pct
0         Top 10%        434  5469382.460      0.100046     0.613751
1          10-20%        434  1180055.001      0.100046     0.132421
2          20-30%        434   728457.380      0.100046     0.081744
3          30-40%        433   490811.450      0.099816     0.055077
4          40-50%        434   344734.671      0.100046     0.038685
5          50-60%        434   252901.481      0.100046     0.028380
6          60-70%        433   179506.281      0.099816     0.020143
7          70-80%        434   132219.390      0.100046     0.014837
8          80-

In [15]:
print((customer_metrics["total_orders"] > 1).sum())
len(customer_metrics)

2845


4338